# Chittagonian ↔ Bengali Neural Machine Translation

## Notebook 03

### mBART-50 Fine-tuning

**Translation Direction**

Standard Bengali → Chatgaya

---

Author : Mostafa Al Moin

Model : facebook/mbart-large-50-many-to-many-mmt

Dataset : 2000 Manual Sentence Pairs

In [ ]:
!pip install -q \
transformers==4.46.3 \
datasets==3.1.0 \
accelerate==1.1.1 \
evaluate==0.4.3 \
sentencepiece==0.2.0 \
sacrebleu==2.4.3 \
peft==0.13.2 \
huggingface-hub==0.26.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 78.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curr

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import transformers
import datasets
import accelerate
import evaluate
import peft

print("Transformers :", transformers.__version__)
print("Datasets     :", datasets.__version__)
print("Accelerate   :", accelerate.__version__)
print("Evaluate     :", evaluate.__version__)
print("PEFT         :", peft.__version__)

Transformers : 4.46.3
Datasets     : 3.1.0
Accelerate   : 1.1.1
Evaluate     : 0.4.3
PEFT         : 0.13.2


In [ ]:
import torch

print("=" * 60)

print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("=" * 60)

CUDA Available : True
GPU : Tesla T4


In [ ]:
import random
import numpy as np
import pandas as pd

from datasets import Dataset

from transformers import (
    MBart50TokenizerFast,
    MBartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed Fixed :", SEED)

Seed Fixed : 42


In [ ]:
train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("validation.csv")

print("="*60)

print("Train Shape :", train_df.shape)
print("Valid Shape :", valid_df.shape)

print("="*60)

display(train_df.head())

Train Shape : (1600, 2)
Valid Shape : (200, 2)


,Bangla,Chatgaya
0,আমি আগে আরো অনেকবার আপনার দোকান থেকে বাজার করেছি,আঁই আগে আরো বতবার অঁনোর দোয়ানত্তুন বাজার গইজ্জি
1,সময়মতো খাবার না খেলে শরীর দুর্বল হয়ে যাবে,সময়মতো হানা ন হাইলি শরীল দুর্বল অই যাইবু
2,"এবার বর্ষা আসার আগে ঘরটা ঠিক করে ফেলতে হবে, না...",এবার বর্ষা আইবের আগে ঘর ইয়েন ঠিক গরি ফেলন ফরিব...
3,আর কত বাটপারি করবি বন্ধু? এবার ভাল হয়ে যা,আর হত বাটফারি গরিবি বন্ধু? এবার ভালা অই যা
4,"আমি ঠিকমতো নাস্তা করতে পারিনি, সেজন্য জলদি খিদ...","আঁই ঠিকগরি নাস্তা গরিত ন ফারি, এতেল্লে হারা ভু..."


In [ ]:
print(train_df.columns)

print()

print(train_df.isnull().sum())

print()

print(valid_df.isnull().sum())

Index(['Bangla', 'Chatgaya'], dtype='object')

Bangla      0
Chatgaya    0
dtype: int64

Bangla      0
Chatgaya    0
dtype: int64


In [ ]:
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

print(train_dataset)
print(valid_dataset)

Dataset({
    features: ['Bangla', 'Chatgaya'],
    num_rows: 1600
})
Dataset({
    features: ['Bangla', 'Chatgaya'],
    num_rows: 200
})


In [ ]:
# ==========================================================
# Load mBART Tokenizer
# ==========================================================

MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)

print("Tokenizer Loaded Successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Tokenizer Loaded Successfully!


In [ ]:
# ==========================================================
# Language Configuration
# ==========================================================

tokenizer.src_lang = "bn_IN"
tokenizer.tgt_lang = "bn_IN"

print("Source Language :", tokenizer.src_lang)
print("Target Language :", tokenizer.tgt_lang)

Source Language : bn_IN
Target Language : bn_IN


In [ ]:
# ==========================================================
# Load mBART Model
# ==========================================================

model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)

print("Model Loaded Successfully!")

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Model Loaded Successfully!


In [ ]:
# ==========================================================
# Configuration
# ==========================================================

MAX_SOURCE_LENGTH = 64
MAX_TARGET_LENGTH = 64

print("Max Source Length :", MAX_SOURCE_LENGTH)
print("Max Target Length :", MAX_TARGET_LENGTH)

Max Source Length : 64
Max Target Length : 64


In [ ]:
# ==========================================================
# Preprocessing Function
# ==========================================================

def preprocess_function(examples):

    model_inputs = tokenizer(
        examples["Bangla"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=examples["Chatgaya"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
# ==========================================================
# Tokenize Dataset
# ==========================================================

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_valid = valid_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=valid_dataset.column_names
)

print(tokenized_train)
print(tokenized_valid)

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1600
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 200
})


In [ ]:
# ==========================================================
# Verify Tokenization
# ==========================================================

sample = tokenized_train[0]

print(sample.keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [ ]:
# ==========================================================
# Decode Source Sentence
# ==========================================================

print(tokenizer.decode(sample["input_ids"], skip_special_tokens=True))

আমি আগে আরো অনেকবার আপনার দোকান থেকে বাজার করেছি


In [ ]:
# ==========================================================
# Decode Target Sentence
# ==========================================================

print(tokenizer.decode(sample["labels"], skip_special_tokens=True))

আঁই আগে আরো বতবার অঁনোর দোয়ানত্তুন বাজার গইজ্জি


In [ ]:
# ==========================================================
# Data Collator
# ==========================================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

print("Data Collator Ready!")

Data Collator Ready!


In [ ]:
# ==========================================================
# Training Arguments
# ==========================================================

training_args = Seq2SeqTrainingArguments(

    output_dir="./mbart_bn2ctg",

    num_train_epochs=10,

    learning_rate=5e-5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    gradient_accumulation_steps=2,

    evaluation_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="steps",

    logging_steps=25,

    save_total_limit=2,

    predict_with_generate=True,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    fp16=torch.cuda.is_available(),

    report_to="none",

    seed=42
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# ==========================================================
# Trainer
# ==========================================================

trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_valid,

    tokenizer=tokenizer,

    data_collator=data_collator
)

print("Trainer Ready!")

/tmp/ipykernel_1223/1158737829.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Trainer Ready!


In [ ]:
# ==========================================================
# Start Training
# ==========================================================

trainer.train()

Epoch,Training Loss,Validation Loss
1,1.180700,1.107509
2,0.727200,0.821372
3,0.413400,0.765547
4,0.280700,0.732446
5,0.140900,0.743726
6,0.084900,0.762306
7,0.046500,0.779061
8,0.033400,0.811511
9,0.021500,0.801300
10,0.010900,0.806041


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:2817: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=2000, training_loss=0.40569878555834293, metrics={'train_runtime': 2375.8153, 'train_samples_per_second': 6.735, 'train_steps_per_second': 0.842, 'total_flos': 731905752367104.0, 'train_loss': 0.40569878555834293, 'epoch': 10.0})

In [ ]:
# ==========================================================
# Save Best Model
# ==========================================================

SAVE_PATH = "mbart_bn2ctg_new"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("✅ Model Saved Successfully!")

✅ Model Saved Successfully!


In [ ]:
import os

print(os.listdir("mbart_bn2ctg_new"))

['sentencepiece.bpe.model', 'tokenizer_config.json', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'config.json', 'training_args.bin', 'generation_config.json']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil

DESTINATION = "/content/drive/MyDrive/Chatgaya-Bangla-NMT/mbart_bn2ctg_new"

shutil.copytree(
    "mbart_bn2ctg_new",
    DESTINATION,
    dirs_exist_ok=True
)

print("✅ Model Backed Up to Google Drive")

✅ Model Backed Up to Google Drive
